# 05 — `abc.ABC`, `@abstractmethod` et `typing.Protocol`

## Objectifs pédagogiques

À la fin de ce notebook, vous saurez :

- définir une classe abstraite avec `abc.ABC` et `@abstractmethod`
- comprendre pourquoi on ne peut pas instancier une classe abstraite
- écrire un `Protocol` pour typer un **comportement** sans imposer l'héritage
- choisir entre `ABC` (héritage nominatif) et `Protocol` (duck typing structurel)
- utiliser `@runtime_checkable` et ses limites

## Prérequis — ce que vous connaissez déjà

À ce stade de la formation intermédiaire, vous maîtrisez :

- classes, héritage, `super()`
- surcharge d'opérateurs
- type hints modernes (`int | None`, `list[T]`)

Ce que nous n'avons **pas encore vu** (et que nous n'utiliserons donc pas dans ce notebook) :

- `TypedDict`, `Literal`, `Final` (notebook 02 jour 2)
- `Generic[T]`, `TypeVar` (notebook 03 jour 2)
- `@dataclass` et son rapport aux ABC (jour 2)

## Plan

1. Pourquoi une interface abstraite ?
2. `abc.ABC` et `@abstractmethod`
3. Interdire l'instanciation directe
4. Le problème de l'héritage obligatoire
5. `typing.Protocol` : duck typing **typé**
6. `@runtime_checkable`
7. ABC vs Protocol : quand choisir ?
8. Pattern : interface + implémentations
9. Synthèse
10. Exercices

---

## 1. Pourquoi une interface abstraite ?

Vous avez déjà écrit plusieurs classes qui partagent un comportement (une méthode `decrire()`, par exemple). Rien ne **garantit** que chaque nouvelle classe le fournira. Une **interface abstraite** pose un contrat : *toute classe qui prétend être une `X` doit fournir `meth1`, `meth2` et `meth3`*.

---

## 2. `abc.ABC` et `@abstractmethod`

Le module `abc` (*abstract base class*) fournit `ABC` et `@abstractmethod`.

In [ ]:
from abc import ABC, abstractmethod


class Salle(ABC):
    @abstractmethod
    def tarif_horaire(self) -> float:
        '''Retourne le tarif horaire en euros.'''

    @abstractmethod
    def capacite(self) -> int:
        '''Retourne la capacité maximale.'''


In [ ]:
class SalleReunion(Salle):
    def __init__(self, capacite: int) -> None:
        self._cap = capacite
    def tarif_horaire(self) -> float:
        return 50.0
    def capacite(self) -> int:
        return self._cap


In [ ]:
SalleReunion(12).tarif_horaire()


---

## 3. Interdire l'instanciation directe

Tant qu'une sous-classe n'implémente pas **toutes** les méthodes abstraites, Python refuse de l'instancier.

In [ ]:
try:
    Salle()
except TypeError as exc:
    print(exc)


In [ ]:
class SalleIncomplete(Salle):
    def tarif_horaire(self) -> float:
        return 30.0
    # oublie capacite()


In [ ]:
try:
    SalleIncomplete()
except TypeError as exc:
    print(exc)


L'erreur arrive à l'instanciation, **pas à la définition** de la classe. Python ne force pas d'implémenter au moment de `class`, il vérifie à `__init__`.

---

## 4. Le problème de l'héritage obligatoire

`ABC` impose d'hériter explicitement. Si vous réutilisez une classe **tierce** qui fournit naturellement les bonnes méthodes mais n'hérite de rien, elle n'est **pas** une `Salle` aux yeux de Python.

In [ ]:
class SalleTiers:  # ne vient pas de notre code
    def tarif_horaire(self) -> float:
        return 80.0
    def capacite(self) -> int:
        return 30


In [ ]:
isinstance(SalleTiers(), Salle)


C'est là qu'entrent en scène les **Protocols**.

---

## 5. `typing.Protocol` : duck typing typé

`Protocol` permet de définir un **contrat structurel** : *quoi que ce soit qui fournit ces méthodes est une `Salle`*, sans avoir besoin d'hériter.

In [ ]:
from typing import Protocol


class Sallable(Protocol):
    def tarif_horaire(self) -> float: ...
    def capacite(self) -> int: ...


In [ ]:
def devis(salle: Sallable, heures: int) -> float:
    return salle.tarif_horaire() * heures


In [ ]:
devis(SalleTiers(), 3)


In [ ]:
devis(SalleReunion(12), 5)


**Aucun des deux ne dérive de `Sallable`**. Tant qu'ils fournissent les bonnes méthodes, `mypy --strict` accepte l'appel. C'est le meilleur des deux mondes : **typage statique** + **duck typing**.

---

## 6. `@runtime_checkable`

Par défaut, un `Protocol` n'est **pas** utilisable avec `isinstance` à l'exécution : ce serait trop coûteux. `@runtime_checkable` active cette vérification (uniquement sur la présence des **attributs**, pas leur signature).

In [ ]:
from typing import Protocol, runtime_checkable


@runtime_checkable
class Printable(Protocol):
    def __str__(self) -> str: ...


In [ ]:
isinstance('hello', Printable)


In [ ]:
isinstance(42, Printable)


⚠️ Cette vérification ne regarde **que** la présence des noms, pas les signatures. C'est volontairement permissif. Pour du typage solide, fiez-vous à `mypy`.

---

## 7. ABC vs Protocol : quand choisir ?

| Critère | `ABC` | `Protocol` |
|---|---|---|
| Relation | Nominale (héritage explicite) | Structurelle (duck typing) |
| Code qu'on contrôle | ✅ | ✅ |
| Code tiers qu'on ne contrôle pas | ❌ | ✅ |
| Peut fournir du code partagé (méthodes concrètes) | ✅ | ❌ |
| `isinstance` | ✅ toujours | ✅ avec `@runtime_checkable` |
| Vérifié par `mypy` | ✅ | ✅ |
| Idiome moderne ? | Oui si on veut imposer l'héritage | **Oui par défaut** pour les interfaces légères |

**Règle simple** : si vous n'avez pas besoin d'imposer un lien d'héritage et pas besoin de fournir du code partagé, utilisez `Protocol`. Sinon, `ABC`.

---

## 8. Pattern : interface + implémentations

L'idiome le plus courant pour rendre votre code **testable** : définir une interface (Protocol), coder contre elle, fournir une implémentation réelle et une de test.

In [ ]:
from typing import Protocol


class BaseDonnees(Protocol):
    def charger_salle(self, nom: str) -> dict: ...
    def sauver_salle(self, nom: str, data: dict) -> None: ...


In [ ]:
class FakeBaseDonnees:
    def __init__(self) -> None:
        self._stockage: dict[str, dict] = {}
    def charger_salle(self, nom: str) -> dict:
        return self._stockage[nom]
    def sauver_salle(self, nom: str, data: dict) -> None:
        self._stockage[nom] = data


In [ ]:
def traiter(db: BaseDonnees) -> None:
    db.sauver_salle('Mars', {'capacite': 12})
    print(db.charger_salle('Mars'))

traiter(FakeBaseDonnees())


---

## Synthèse

| Outil | Import | Rôle |
|---|---|---|
| `ABC` | `from abc import ABC` | Classe de base pour interfaces nominales |
| `@abstractmethod` | `from abc import abstractmethod` | Marque une méthode comme à implémenter |
| `Protocol` | `from typing import Protocol` | Interface structurelle (duck typing typé) |
| `@runtime_checkable` | `from typing import runtime_checkable` | Autorise `isinstance` sur un Protocol |


### Règles à retenir

1. **`ABC` impose l'héritage**, `Protocol` non. Choisissez selon que vous voulez contraindre ou déclarer un contrat.
2. **Une `ABC` avec toutes ses méthodes abstraites ne peut pas être instanciée** — Python le vérifie à l'instanciation.
3. **Pour tester des classes qui dépendent d'un service**, définissez un `Protocol` et injectez un fake.
4. **`@runtime_checkable` est permissif** : utile pour un check rapide, pas pour du code critique.
5. **Le duck typing vit très bien sans annotation** ; `Protocol` le rend **explicite** et vérifiable par `mypy`.

---

## Exercices

Les exercices sont gradués. Tous utilisent des fonctions typées (PEP 604).

### Exercice 1 — `Forme` abstraite *(facile)*

Définir une classe abstraite `Forme(ABC)` avec deux méthodes abstraites : `aire(self) -> float` et `perimetre(self) -> float`. Implémenter `Carre(cote)` et `Cercle(rayon)` qui héritent de `Forme`. Vérifier qu'instancier `Forme()` échoue.

In [ ]:
# Votre code ici


In [ ]:
# ▶ Une fois votre solution écrite ci-dessus, exécutez cette cellule
# pour signaler à votre formateur que vous avez tenté l'exercice.
import sys
from pathlib import Path
for _p in (Path.cwd(), *Path.cwd().parents):
    if (_p / "_common" / "utils_pedagogie.py").exists():
        sys.path.insert(0, str(_p / "_common")); break
from utils_pedagogie import marquer_tentative
marquer_tentative(notebook="05_Abc_et_protocol", exercice=1)


<details>
<summary>📖 Voir la correction</summary>

```python
from abc import ABC, abstractmethod
import math

class Forme(ABC):
    @abstractmethod
    def aire(self) -> float: ...
    @abstractmethod
    def perimetre(self) -> float: ...

class Carre(Forme):
    def __init__(self, cote: float) -> None:
        self.cote = cote
    def aire(self) -> float:
        return self.cote ** 2
    def perimetre(self) -> float:
        return 4 * self.cote

class Cercle(Forme):
    def __init__(self, rayon: float) -> None:
        self.rayon = rayon
    def aire(self) -> float:
        return math.pi * self.rayon ** 2
    def perimetre(self) -> float:
        return 2 * math.pi * self.rayon

try:
    Forme()
except TypeError as exc:
    print(exc)
print(Carre(3).aire(), Cercle(1).perimetre())
```

</details>

### Exercice 2 — Protocol `Describable` *(moyen)*

Définir un `Protocol` `Describable` avec une seule méthode `decrire(self) -> str`. Écrire une fonction `afficher(objets: list[Describable]) -> None` qui affiche la description de chaque objet. Tester avec deux classes totalement indépendantes (pas d'héritage entre elles, pas d'héritage de `Describable`).

In [ ]:
# Votre code ici


In [ ]:
# ▶ Une fois votre solution écrite ci-dessus, exécutez cette cellule
# pour signaler à votre formateur que vous avez tenté l'exercice.
import sys
from pathlib import Path
for _p in (Path.cwd(), *Path.cwd().parents):
    if (_p / "_common" / "utils_pedagogie.py").exists():
        sys.path.insert(0, str(_p / "_common")); break
from utils_pedagogie import marquer_tentative
marquer_tentative(notebook="05_Abc_et_protocol", exercice=2)


<details>
<summary>📖 Voir la correction</summary>

```python
from typing import Protocol

class Describable(Protocol):
    def decrire(self) -> str: ...

class Livre:
    def __init__(self, titre: str) -> None:
        self.titre = titre
    def decrire(self) -> str:
        return f'Livre : {self.titre}'

class Voiture:
    def __init__(self, marque: str) -> None:
        self.marque = marque
    def decrire(self) -> str:
        return f'Voiture : {self.marque}'

def afficher(objets: list[Describable]) -> None:
    for obj in objets:
        print(obj.decrire())

afficher([Livre('1984'), Voiture('Peugeot')])
```

</details>

### Exercice 3 — ABC avec code partagé *(moyen)*

Écrire une `ABC` `Salle` qui fournit un attribut `nom` dans `__init__`, une **méthode concrète** `decrire()` qui utilise `tarif_horaire()` (abstraite) pour afficher le nom + tarif, et une méthode abstraite `tarif_horaire()`. Implémenter `SalleStandard` et `SalleVIP` avec des tarifs différents.

In [ ]:
# Votre code ici


In [ ]:
# ▶ Une fois votre solution écrite ci-dessus, exécutez cette cellule
# pour signaler à votre formateur que vous avez tenté l'exercice.
import sys
from pathlib import Path
for _p in (Path.cwd(), *Path.cwd().parents):
    if (_p / "_common" / "utils_pedagogie.py").exists():
        sys.path.insert(0, str(_p / "_common")); break
from utils_pedagogie import marquer_tentative
marquer_tentative(notebook="05_Abc_et_protocol", exercice=3)


<details>
<summary>📖 Voir la correction</summary>

```python
from abc import ABC, abstractmethod

class Salle(ABC):
    def __init__(self, nom: str) -> None:
        self.nom = nom
    @abstractmethod
    def tarif_horaire(self) -> float: ...
    def decrire(self) -> str:
        return f'{self.nom} : {self.tarif_horaire():.2f} €/h'

class SalleStandard(Salle):
    def tarif_horaire(self) -> float:
        return 50.0

class SalleVIP(Salle):
    def tarif_horaire(self) -> float:
        return 200.0

print(SalleStandard('Mars').decrire())
print(SalleVIP('Olympus').decrire())
```

</details>

### Exercice 4 — Injection de dépendance avec Protocol *(difficile)*

Définir un `Protocol` `Notifieur` avec `envoyer(self, message: str) -> None`. Écrire deux implémentations : `NotifieurConsole` (print) et `NotifieurFake` (stocke les messages dans une liste). Écrire une classe `Service` dont le constructeur reçoit un `Notifieur` et qui expose `alerte(self, msg: str) -> None`. Tester le `Service` avec le fake, puis vérifier que la liste contient bien le message.

In [ ]:
# Votre code ici


In [ ]:
# ▶ Une fois votre solution écrite ci-dessus, exécutez cette cellule
# pour signaler à votre formateur que vous avez tenté l'exercice.
import sys
from pathlib import Path
for _p in (Path.cwd(), *Path.cwd().parents):
    if (_p / "_common" / "utils_pedagogie.py").exists():
        sys.path.insert(0, str(_p / "_common")); break
from utils_pedagogie import marquer_tentative
marquer_tentative(notebook="05_Abc_et_protocol", exercice=4)


<details>
<summary>📖 Voir la correction</summary>

```python
from typing import Protocol

class Notifieur(Protocol):
    def envoyer(self, message: str) -> None: ...

class NotifieurConsole:
    def envoyer(self, message: str) -> None:
        print(f'[CONSOLE] {message}')

class NotifieurFake:
    def __init__(self) -> None:
        self.envois: list[str] = []
    def envoyer(self, message: str) -> None:
        self.envois.append(message)

class Service:
    def __init__(self, notifieur: Notifieur) -> None:
        self.notifieur = notifieur
    def alerte(self, msg: str) -> None:
        self.notifieur.envoyer(f'ALERTE : {msg}')

fake = NotifieurFake()
Service(fake).alerte('disque plein')
assert fake.envois == ['ALERTE : disque plein']
print('ok')
```

</details>

---

## Ressources externes

### Documentation officielle
- [`abc` — Abstract Base Classes](https://docs.python.org/3/library/abc.html)
- [`typing.Protocol`](https://docs.python.org/3/library/typing.html#typing.Protocol)

### PEPs de référence
- **PEP 544** — *Protocols: Structural subtyping*
- **PEP 3119** — *Introducing ABCs*

### Lectures complémentaires
- Glyph Lefkowitz, *Duck Typing, Protocols and Python* (2020).